---
Accession Extraction Validation - Evaluation Component

This module handles the evaluation of model predictions against ground truth.
It computes metrics, statistical tests, and generates reports.

Project: AI6129 Pathogen Tracking System
Version: 1.1
---

In [1]:
import json
import re
from datetime import datetime
import pandas as pd
import numpy as np
from dataclasses import dataclass, field
from enum import Enum
from pathlib import Path
from typing import Callable, Dict, List, Set, Tuple, Optional, Any
from scipy import stats
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive', force_remount=False)


Mounted at /content/drive


---
Type Definitions
---

In [2]:
class Stratum(Enum):
    """Accession density categories matching validation_splits.json"""
    ZERO = "zero"
    LOW = "low"
    MEDIUM = "medium"
    HIGH = "high"


@dataclass
class GroundTruthDocument:
    """Represents a single document from ground truth data"""
    pmcid: str
    accessions: Set[str]
    stratum: Stratum
    accession_count: int


@dataclass
class PredictionDocument:
    """Represents model predictions for a single document"""
    pmcid: str
    extracted_accessions: Set[str]
    raw_model_output: Optional[str]
    processing_time_ms: Optional[int]


@dataclass
class DocumentMetrics:
    """Computed metrics for a single document"""
    pmcid: str
    stratum: Stratum
    true_positives: int
    false_positives: int
    false_negatives: int
    precision: float
    recall: float
    f1: float
    is_exact_match: bool
    has_hallucination: bool


@dataclass
class AggregateMetrics:
    """Aggregated metrics over a collection of documents"""
    document_count: int
    macro_precision: float
    macro_recall: float
    macro_f1: float
    micro_precision: float
    micro_recall: float
    micro_f1: float
    exact_match_rate: float
    hallucination_rate: float
    total_true_positives: int
    total_false_positives: int
    total_false_negatives: int


@dataclass
class ConfidenceInterval:
    """Bootstrap confidence interval for a metric"""
    point_estimate: float
    lower_bound: float
    upper_bound: float
    confidence_level: float


@dataclass
class McNemarResult:
    """Result of McNemar's test comparing two models"""
    model_a: str
    model_b: str
    test_type: str
    n_both_correct: int
    n_both_wrong: int
    n_a_only_correct: int
    n_b_only_correct: int
    statistic: float
    p_value: float
    is_significant: bool
    adjusted_alpha: float


@dataclass
class StratifiedReport:
    """Metrics for a single stratum"""
    stratum: Stratum
    metrics: AggregateMetrics
    document_count: int


@dataclass
class EvaluationReport:
    """Complete evaluation report for a single model"""
    model_id: str
    split_name: str
    overall_metrics: AggregateMetrics
    positive_only_metrics: AggregateMetrics
    stratified_reports: List[StratifiedReport]
    confidence_intervals: Dict[str, ConfidenceInterval]
    document_metrics: List[DocumentMetrics]

---
NORMALISATION FUNCTIONS
---

In [3]:
def normalise_accession(accession: str) -> str:
    """
    Normalise a single accession number for consistent comparison.

    Parameters:
        accession: Raw accession string

    Returns:
        Normalised accession (uppercase, trimmed, whitespace removed, suffix stripped)
    """
    # v1.1 #changed - Added whitespace removal to handle cases like "PRJNA\n813213"
    normalised = re.sub(r'\s+', '', accession)  # v1.1 #changed
    # v1.2 #changed - Strip [GenBank] or similar database suffixes
    normalised = re.sub(r'\[.*?\]', '', normalised)  # v1.2 #changed
    normalised = normalised.upper()
    normalised = normalised.strip()
    return normalised


def normalise_accession_set(accessions: List[str]) -> Set[str]:
    """
    Normalise a list of accessions to a set.

    Parameters:
        accessions: List of raw accession strings

    Returns:
        Set of normalised accessions
    """
    normalised_set = set()

    for acc in accessions:
        # v1.1 #changed - Added empty string check per pseudo-code
        if acc is None or acc == '':  # v1.1 #changed
            continue
        normalised = normalise_accession(acc)
        # v1.1 #changed - Check if normalisation results in empty string
        if normalised:  # v1.1 #changed
            normalised_set.add(normalised)

    return normalised_set

---
Data Loading Functions
---

In [4]:
def load_ground_truth(ground_truth_filepath: str, split_name: str
                      ) -> Dict[str, GroundTruthDocument]:
    """
    Load ground truth data for a specific split.

    Parameters:
        ground_truth_filepath: Path to validation_splits.json
        split_name: Name of split to load (development/validation/test)

    Returns:
        Mapping of PMCID -> GroundTruthDocument
    """
    ground_truth_docs = {}

    with open(ground_truth_filepath, 'r', encoding='utf-8') as f:
        data = json.load(f)

    split_data = data['splits'].get(split_name, {})
    split_pmcids = split_data.get('files', []) if isinstance(split_data, dict) else split_data

    document_details = data.get('document_details', {})

    for pmcid in split_pmcids:
        if pmcid not in document_details:
            continue

        doc_info = document_details[pmcid]
        accessions_raw = doc_info.get('accessions', [])
        accessions_normalised = normalise_accession_set(accessions_raw)
        stratum_str = doc_info.get('stratum', 'zero')
        stratum = Stratum(stratum_str)
        accession_count = doc_info.get('accession_count', 0)

        ground_truth_docs[pmcid] = GroundTruthDocument(
            pmcid=pmcid,
            accessions=accessions_normalised,
            stratum=stratum,
            accession_count=accession_count
        )

    return ground_truth_docs


def load_predictions(predictions_filepath: str
                     ) -> Tuple[Dict[str, PredictionDocument], Dict[str, Any]]:
    """
    Load model predictions from experiment output file.

    Parameters:
        predictions_filepath: Path to predictions JSON file

    Returns:
        Tuple of (PMCID -> PredictionDocument mapping, experiment metadata)
    """
    prediction_docs = {}

    with open(predictions_filepath, 'r', encoding='utf-8') as f:
        data = json.load(f)

    metadata = data.get('experiment_metadata', {})
    predictions_list = data.get('predictions', [])

    for pred in predictions_list:
        pmcid = pred.get('pmcid', '')
        if not pmcid:
            continue

        accessions_raw = pred.get('extracted_accessions', [])
        accessions_normalised = normalise_accession_set(accessions_raw)
        raw_output = pred.get('raw_model_output')
        processing_time = pred.get('processing_time_ms')

        prediction_docs[pmcid] = PredictionDocument(
            pmcid=pmcid,
            extracted_accessions=accessions_normalised,
            raw_model_output=raw_output,
            processing_time_ms=processing_time
        )

    return prediction_docs, metadata


def load_split_pmcids(ground_truth_filepath: str,
                      split_name: str
                      ) -> List[str]:
    """
    Get list of PMCIDs for a specific split.

    Parameters:
        ground_truth_filepath: Path to validation_splits.json
        split_name: Name of split

    Returns:
        List of PMCIDs in the split
    """
    with open(ground_truth_filepath, 'r', encoding='utf-8') as f:
        data = json.load(f)

    split_data = data['splits'].get(split_name, {})
    # v1.1 #changed - Fixed typo: 'instance' -> 'isinstance'
    return split_data.get('files', []) if isinstance(split_data, dict) else split_data  # v1.1 #changed

---
METRIC COMPUTATION FUNCTIONS
---

In [5]:
def compute_set_overlap(
    predicted: Set[str],
    ground_truth: Set[str]
) -> Tuple[int, int, int]:
    """
    Calculate set overlap statistics.

    Parameters:
        predicted: Set of predicted accessions
        ground_truth: Set of ground truth accessions

    Returns:
        Tuple of (true_positives, false_positives, false_negatives)
    """
    true_positives = len(predicted & ground_truth)
    false_positives = len(predicted - ground_truth)
    false_negatives = len(ground_truth - predicted)

    return true_positives, false_positives, false_negatives


def compute_precision(true_positives: int, false_positives: int) -> float:
    """Calculate precision: TP / (TP + FP)"""
    denominator = true_positives + false_positives
    if denominator == 0:
        return 1.0
    return true_positives / denominator


def compute_recall(true_positives: int, false_negatives: int) -> float:
    """Calculate recall: TP / (TP + FN)"""
    denominator = true_positives + false_negatives
    if denominator == 0:
        return 1.0
    return true_positives / denominator


def compute_f1(precision: float, recall: float) -> float:
    """Calculate F1 score: harmonic mean of precision and recall"""
    if precision + recall == 0:
        return 0.0
    return 2 * (precision * recall) / (precision + recall)


def compute_document_metrics(
    prediction: PredictionDocument,
    ground_truth: GroundTruthDocument
) -> DocumentMetrics:
    """
    Compute all metrics for a single document.

    Parameters:
        prediction: Model prediction for document
        ground_truth: Ground truth for document

    Returns:
        DocumentMetrics with all computed values
    """
    tp, fp, fn = compute_set_overlap(
        prediction.extracted_accessions,
        ground_truth.accessions
    )

    precision = compute_precision(tp, fp)
    recall = compute_recall(tp, fn)
    f1 = compute_f1(precision, recall)

    is_exact = (prediction.extracted_accessions == ground_truth.accessions)
    has_hallucination = (fp > 0)

    return DocumentMetrics(
        pmcid=ground_truth.pmcid,
        stratum=ground_truth.stratum,
        true_positives=tp,
        false_positives=fp,
        false_negatives=fn,
        precision=precision,
        recall=recall,
        f1=f1,
        is_exact_match=is_exact,
        has_hallucination=has_hallucination
    )


def compute_aggregate_metrics(
    document_metrics_list: List[DocumentMetrics]
) -> AggregateMetrics:
    """
    Aggregate metrics across multiple documents.

    Parameters:
        document_metrics_list: List of per-document metrics

    Returns:
        AggregateMetrics with macro and micro averages
    """
    if not document_metrics_list:
        return AggregateMetrics(
            document_count=0,
            macro_precision=0.0,
            macro_recall=0.0,
            macro_f1=0.0,
            micro_precision=0.0,
            micro_recall=0.0,
            micro_f1=0.0,
            exact_match_rate=0.0,
            hallucination_rate=0.0,
            total_true_positives=0,
            total_false_positives=0,
            total_false_negatives=0
        )

    document_count = len(document_metrics_list)

    # Macro averages
    macro_precision = sum(m.precision for m in document_metrics_list) / document_count
    macro_recall = sum(m.recall for m in document_metrics_list) / document_count
    # v1.1 #changed - Fixed: was missing division by document_count
    macro_f1 = sum(m.f1 for m in document_metrics_list) / document_count  # v1.1 #changed

    # v1.1 #changed - Fixed: was summing wrong fields (precision/recall instead of tp/fp)
    # Totals for micro averages
    total_tp = sum(m.true_positives for m in document_metrics_list)  # v1.1 #changed
    total_fp = sum(m.false_positives for m in document_metrics_list)  # v1.1 #changed
    total_fn = sum(m.false_negatives for m in document_metrics_list)

    # Micro averages
    micro_precision = compute_precision(total_tp, total_fp)
    micro_recall = compute_recall(total_tp, total_fn)
    micro_f1 = compute_f1(micro_precision, micro_recall)

    # Rates
    exact_match_count = sum(1 for m in document_metrics_list if m.is_exact_match)
    hallucination_count = sum(1 for m in document_metrics_list if m.has_hallucination)

    # v1.1 #changed - Fixed: rates now properly divided by document_count
    exact_match_rate = exact_match_count / document_count  # v1.1 #changed
    hallucination_rate = hallucination_count / document_count  # v1.1 #changed

    return AggregateMetrics(
        document_count=document_count,
        macro_precision=macro_precision,
        macro_recall=macro_recall,
        macro_f1=macro_f1,
        micro_precision=micro_precision,
        micro_recall=micro_recall,
        micro_f1=micro_f1,
        exact_match_rate=exact_match_rate,
        hallucination_rate=hallucination_rate,
        total_true_positives=total_tp,
        total_false_positives=total_fp,
        total_false_negatives=total_fn
    )


def filter_positive_documents(document_metrics_list: List[DocumentMetrics],
                              ground_truth_docs: Dict[str, GroundTruthDocument]
                              ) -> List[DocumentMetrics]:
    """
    Filter to documents that have at least one ground truth accession.

    Parameters:
        document_metrics_list: All document metrics
        ground_truth_docs: Ground truth lookup

    Returns:
        Filtered list with only positive documents
    """
    positive_docs = []

    for doc_metrics in document_metrics_list:
        gt_doc = ground_truth_docs.get(doc_metrics.pmcid)
        if gt_doc and len(gt_doc.accessions) > 0:
            positive_docs.append(doc_metrics)

    return positive_docs


def filter_by_stratum(
    document_metrics_list: List[DocumentMetrics],
    stratum: Stratum
) -> List[DocumentMetrics]:
    """
    Filter documents by density stratum.

    Parameters:
        document_metrics_list: All document metrics
        stratum: Target stratum to filter for

    Returns:
        Filtered list with only matching stratum
    """
    return [m for m in document_metrics_list if m.stratum == stratum]


def compute_stratified_metrics(document_metrics_list: List[DocumentMetrics]
                               ) -> List[StratifiedReport]:
    """
    Compute metrics for each density stratum.

    Parameters:
        document_metrics_list: All document metrics

    Returns:
        List of StratifiedReport for each stratum
    """
    reports = []

    for layer in Stratum:
        stratum_docs = filter_by_stratum(document_metrics_list, layer)

        if not stratum_docs:
            continue

        metrics = compute_aggregate_metrics(stratum_docs)
        reports.append(StratifiedReport(
            stratum=layer,
            metrics=metrics,
            document_count=len(stratum_docs)
        ))
    return reports

---
STATISTICAL TESTS (McNemar)
---

In [6]:
def build_mcnemar_contingency_table(metrics_a: List[DocumentMetrics],
                                    metrics_b: List[DocumentMetrics],
                                    test_type: str
                                    ) -> Tuple[int, int, int, int]:
    """
    Build 2x2 contingency table for McNemar's test.

    Parameters:
        metrics_a: Document metrics from model A
        metrics_b: Document metrics from model B
        test_type: "exact_match" or "hallucination"

    Returns:
        Tuple of (both_correct, both_wrong, a_only_correct, b_only_correct)
    """
    metrics_a_dict = {m.pmcid: m for m in metrics_a}
    metrics_b_dict = {m.pmcid: m for m in metrics_b}

    common_pmcids = set(metrics_a_dict.keys()) & set(metrics_b_dict.keys())

    both_correct = 0
    both_wrong = 0
    # v1.1 #changed - Fixed variable names to match usage
    a_only_correct = 0  # v1.1 #changed
    b_only_correct = 0  # v1.1 #changed

    for pmcid in common_pmcids:
        m_a = metrics_a_dict[pmcid]
        m_b = metrics_b_dict[pmcid]

        if test_type == "exact_match":
            a_success = m_a.is_exact_match
            b_success = m_b.is_exact_match
        else:  # hallucination test
            # v1.1 #changed - success means NO hallucination
            a_success = not m_a.has_hallucination  # v1.1 #changed
            b_success = not m_b.has_hallucination  # v1.1 #changed

        # v1.1 #changed - Fixed contingency table logic completely
        if a_success and b_success:
            both_correct += 1
        elif (not a_success) and (not b_success):  # v1.1 #changed
            both_wrong += 1
        elif a_success and (not b_success):  # v1.1 #changed
            a_only_correct += 1  # v1.1 #changed
        else:  # b_success and (not a_success)
            b_only_correct += 1  # v1.1 #changed

    return both_correct, both_wrong, a_only_correct, b_only_correct


def compute_mcnemar_test(
    metrics_a: List[DocumentMetrics],
    metrics_b: List[DocumentMetrics],
    model_a_id: str,
    model_b_id: str,
    test_type: str,
    alpha: float = 0.05
) -> McNemarResult:
    """
    Perform McNemar's test comparing two models.

    Parameters:
        metrics_a: Document metrics from model A
        metrics_b: Document metrics from model B
        model_a_id: Identifier for model A
        model_b_id: Identifier for model B
        test_type: "exact_match" or "hallucination"
        alpha: Significance level

    Returns:
        McNemarResult with test statistics
    """
    both_correct, both_wrong, a_only, b_only = build_mcnemar_contingency_table(
        metrics_a, metrics_b, test_type
    )
    n_discordant = a_only + b_only

    if n_discordant == 0:
        statistic = 0.0
        p_value = 1.0
    elif n_discordant < 25:
        # Exact test using binomial distribution
        result = stats.binomtest(min(a_only, b_only), n_discordant, 0.5)
        statistic = float(a_only)
        p_value = result.pvalue
    else:
        # Chi-squared approximation with Yates continuity correction
        statistic = ((abs(a_only - b_only) - 1) ** 2) / (a_only + b_only)
        p_value = 1 - stats.chi2.cdf(statistic, df=1)

    is_significant = p_value < alpha

    return McNemarResult(
        model_a=model_a_id,
        model_b=model_b_id,
        test_type=test_type,
        n_both_correct=both_correct,
        n_both_wrong=both_wrong,
        n_a_only_correct=a_only,
        n_b_only_correct=b_only,
        statistic=statistic,
        p_value=p_value,
        is_significant=is_significant,
        adjusted_alpha=alpha
    )


def apply_holm_bonferroni_correction(
    results: List[McNemarResult],
    alpha: float = 0.05
) -> List[McNemarResult]:
    """
    Apply Holm-Bonferroni correction for multiple comparisons.

    Parameters:
        results: List of McNemar test results
        alpha: Family-wise error rate

    Returns:
        Updated results with corrected significance
    """
    if not results:
        return results

    n_tests = len(results)
    sorted_results = sorted(results, key=lambda r: r.p_value)

    corrected_results = []
    # v1.1 #changed - Track if any test has failed for step-down procedure
    any_failed = False  # v1.1 #changed

    for i, result in enumerate(sorted_results):
        # v1.1 #changed - Fixed formula: alpha / (n_tests - i) per pseudo-code
        adjusted_alpha = alpha / (n_tests - i)  # v1.1 #changed

        # v1.1 #changed - Implement step-down procedure
        if any_failed:  # v1.1 #changed
            is_significant = False  # v1.1 #changed
        else:  # v1.1 #changed
            is_significant = result.p_value < adjusted_alpha  # v1.1 #changed
            if not is_significant:  # v1.1 #changed
                any_failed = True  # v1.1 #changed

        corrected_result = McNemarResult(
            model_a=result.model_a,
            model_b=result.model_b,
            test_type=result.test_type,
            n_both_correct=result.n_both_correct,
            n_both_wrong=result.n_both_wrong,
            n_a_only_correct=result.n_a_only_correct,
            n_b_only_correct=result.n_b_only_correct,
            statistic=result.statistic,
            p_value=result.p_value,
            is_significant=is_significant,
            adjusted_alpha=adjusted_alpha
        )
        corrected_results.append(corrected_result)

    return corrected_results

---
CONFIDENCE INTERVALS (Bootstrap)
---

In [7]:
def bootstrap_metric(document_metrics_list: List[DocumentMetrics],
                     metric_fn: Callable[[List[DocumentMetrics]], float],
                     n_bootstrap: int = 1000,
                     confidence_level: float = 0.95,
                     random_seed: int = 42
                    ) -> ConfidenceInterval:
    """
    Compute bootstrap confidence interval for a metric.

    Parameters:
        document_metrics_list: List of document metrics
        metric_fn: Function that computes metric from list of DocumentMetrics
        n_bootstrap: Number of bootstrap samples
        confidence_level: Confidence level (e.g., 0.95)
        random_seed: Random seed for reproducibility

    Returns:
        ConfidenceInterval with bounds
    """
    np.random.seed(random_seed)

    # v1.1 #changed - Fixed: defined n_docs variable properly
    n_docs = len(document_metrics_list)  # v1.1 #changed

    # v1.1 #changed - Fixed: was 'n.docs' which is invalid syntax
    if n_docs == 0:  # v1.1 #changed
        return ConfidenceInterval(
            point_estimate=0.0,
            lower_bound=0.0,
            upper_bound=0.0,
            confidence_level=confidence_level
        )

    point_estimate = metric_fn(document_metrics_list)

    bootstrap_values = []

    for _ in range(n_bootstrap):
        # v1.1 #changed - Fixed typo: 'radint' -> 'randint'
        indices = np.random.randint(0, n_docs, size=n_docs)  # v1.1 #changed
        sample = [document_metrics_list[i] for i in indices]
        bootstrap_values.append(metric_fn(sample))

    alpha = 1 - confidence_level
    lower_percentile = (alpha / 2) * 100
    # v1.1 #changed - Fixed parentheses for correct calculation
    upper_percentile = (1 - alpha / 2) * 100  # v1.1 #changed

    lower_bound = float(np.percentile(bootstrap_values, lower_percentile))
    upper_bound = float(np.percentile(bootstrap_values, upper_percentile))

    return ConfidenceInterval(
        point_estimate=point_estimate,
        lower_bound=lower_bound,
        upper_bound=upper_bound,
        confidence_level=confidence_level
    )


def compute_confidence_intervals(document_metrics_list: List[DocumentMetrics],
                                 n_bootstrap: int = 1000,
                                 confidence_level: float = 0.95
                                ) -> Dict[str, ConfidenceInterval]:
    """
    Compute confidence intervals for key metrics.

    Parameters:
        document_metrics_list: List of document metrics
        n_bootstrap: Number of bootstrap samples
        confidence_level: Confidence level

    Returns:
        Dictionary mapping metric name to ConfidenceInterval
    """
    intervals = {}

    # Macro F1
    def macro_f1_fn(docs):
        if not docs:
            return 0.0
        return sum(m.f1 for m in docs) / len(docs)

    intervals['macro_f1'] = bootstrap_metric(
        document_metrics_list, macro_f1_fn, n_bootstrap, confidence_level
    )

    # Exact match rate
    def exact_match_fn(docs):
        if not docs:
            return 0.0
        return sum(1 for m in docs if m.is_exact_match) / len(docs)

    intervals['exact_match_rate'] = bootstrap_metric(
        document_metrics_list, exact_match_fn, n_bootstrap, confidence_level
    )

    # Hallucination rate
    def hallucination_fn(docs):
        if not docs:
            return 0.0
        return sum(1 for m in docs if m.has_hallucination) / len(docs)

    intervals['hallucination_rate'] = bootstrap_metric(
        document_metrics_list, hallucination_fn, n_bootstrap, confidence_level
    )

    return intervals

---
MAIN EVALUATION PIPELINE
---

In [8]:
def evaluate_single_model(predictions_filepath: str,
                          ground_truth_filepath: str,
                          split_name: str
                        ) -> EvaluationReport:
    """
    Perform full evaluation for a single model.

    Parameters:
        predictions_filepath: Path to model predictions
        ground_truth_filepath: Path to ground truth
        split_name: Data split name

    Returns:
        Complete EvaluationReport
    """
    predictions, metadata = load_predictions(predictions_filepath)
    ground_truth_docs = load_ground_truth(ground_truth_filepath, split_name)

    model_id = metadata.get('model_id', 'unknown')

    # Compute per-document metrics
    document_metrics_list = []

    for pmcid, gt_doc in ground_truth_docs.items():
        if pmcid not in predictions:
            pred_doc = PredictionDocument(
                pmcid=pmcid,
                extracted_accessions=set(),
                raw_model_output=None,
                processing_time_ms=None
            )
        else:
            pred_doc = predictions[pmcid]

        doc_metrics = compute_document_metrics(pred_doc, gt_doc)
        document_metrics_list.append(doc_metrics)

    # Overall metrics
    overall_metrics = compute_aggregate_metrics(document_metrics_list)

    # Positive-only metrics
    positive_docs = filter_positive_documents(document_metrics_list, ground_truth_docs)
    positive_only_metrics = compute_aggregate_metrics(positive_docs)

    # Stratified metrics
    stratified_reports = compute_stratified_metrics(document_metrics_list)

    # Confidence intervals
    confidence_intervals = compute_confidence_intervals(document_metrics_list)

    return EvaluationReport(
        model_id=model_id,
        split_name=split_name,
        overall_metrics=overall_metrics,
        positive_only_metrics=positive_only_metrics,
        stratified_reports=stratified_reports,
        confidence_intervals=confidence_intervals,
        document_metrics=document_metrics_list
    )


def compare_models(reports: List[EvaluationReport],
                   alpha: float = 0.05
                ) -> List[McNemarResult]:
    """
    Perform pairwise model comparisons using McNemar's test.

    Parameters:
        reports: List of evaluation reports for different models
        alpha: Family-wise error rate

    Returns:
        List of McNemar test results with Holm-Bonferroni correction
    """
    all_results = []

    for i in range(len(reports)):
        for j in range(i + 1, len(reports)):
            report_a = reports[i]
            report_b = reports[j]

            # Exact match comparison
            em_result = compute_mcnemar_test(
                report_a.document_metrics,
                report_b.document_metrics,
                report_a.model_id,
                report_b.model_id,
                "exact_match",
                alpha
            )
            all_results.append(em_result)

            # Hallucination comparison
            hal_result = compute_mcnemar_test(
                report_a.document_metrics,
                report_b.document_metrics,
                report_a.model_id,
                report_b.model_id,
                "hallucination",
                alpha
            )
            all_results.append(hal_result)

    corrected_results = apply_holm_bonferroni_correction(all_results, alpha)

    return corrected_results

---
REPORT GENERATION
---

In [9]:
def generate_summary_table(reports: List[EvaluationReport]
                           ) -> pd.DataFrame:
    """
    Generate summary table comparing all models.

    Parameters:
        reports: List of evaluation reports

    Returns:
        DataFrame with model comparison
    """
    rows = []

    for report in reports:
        row = {
            'model_id': report.model_id,
            'split': report.split_name,
            'n_documents': report.overall_metrics.document_count,
            'macro_precision': report.overall_metrics.macro_precision,
            'macro_recall': report.overall_metrics.macro_recall,
            'macro_f1': report.overall_metrics.macro_f1,
            'micro_f1': report.overall_metrics.micro_f1,
            'exact_match_rate': report.overall_metrics.exact_match_rate,
            'hallucination_rate': report.overall_metrics.hallucination_rate,
            'f1_ci_lower': report.confidence_intervals['macro_f1'].lower_bound,
            'f1_ci_upper': report.confidence_intervals['macro_f1'].upper_bound
        }
        rows.append(row)

    return pd.DataFrame(rows)


def generate_stratified_table(reports: List[EvaluationReport]
                              ) -> pd.DataFrame:
    """
    Generate stratified performance table.

    Parameters:
        reports: List of evaluation reports

    Returns:
        DataFrame with stratified metrics
    """
    rows = []

    for report in reports:
        for strat_report in report.stratified_reports:
            row = {
                'model_id': report.model_id,
                'stratum': strat_report.stratum.value,
                'n_documents': strat_report.document_count,
                'macro_f1': strat_report.metrics.macro_f1,
                'exact_match_rate': strat_report.metrics.exact_match_rate,
                'hallucination_rate': strat_report.metrics.hallucination_rate
            }
            rows.append(row)

    return pd.DataFrame(rows)


def generate_mcnemar_table(results: List[McNemarResult]
                           ) -> pd.DataFrame:
    """
    Generate table of McNemar test results.

    Parameters:
        results: List of McNemar test results

    Returns:
        DataFrame with statistical comparisons
    """
    rows = []

    for result in results:
        row = {
            'model_a': result.model_a,
            'model_b': result.model_b,
            'test_type': result.test_type,
            'n_discordant': result.n_a_only_correct + result.n_b_only_correct,
            'a_only_correct': result.n_a_only_correct,
            'b_only_correct': result.n_b_only_correct,
            'statistic': result.statistic,
            'p_value': result.p_value,
            'adjusted_alpha': result.adjusted_alpha,
            'is_significant': result.is_significant
        }
        rows.append(row)

    return pd.DataFrame(rows)


def save_evaluation_report(
    report: EvaluationReport,
    output_filepath: str
) -> None:
    """
    Save evaluation report to JSON file.

    Parameters:
        report: EvaluationReport to save
        output_filepath: Path for output file
    """
    def metrics_to_dict(m: AggregateMetrics) -> dict:
        return {
            'document_count': m.document_count,
            'macro_precision': m.macro_precision,
            'macro_recall': m.macro_recall,
            'macro_f1': m.macro_f1,
            'micro_precision': m.micro_precision,
            'micro_recall': m.micro_recall,
            'micro_f1': m.micro_f1,
            'exact_match_rate': m.exact_match_rate,
            'hallucination_rate': m.hallucination_rate,
            'total_true_positives': m.total_true_positives,
            'total_false_positives': m.total_false_positives,
            'total_false_negatives': m.total_false_negatives
        }

    def ci_to_dict(ci: ConfidenceInterval) -> dict:
        return {
            'point_estimate': ci.point_estimate,
            'lower_bound': ci.lower_bound,
            'upper_bound': ci.upper_bound,
            'confidence_level': ci.confidence_level
        }

    output_data = {
        'model_id': report.model_id,
        'split_name': report.split_name,
        'overall_metrics': metrics_to_dict(report.overall_metrics),
        'positive_only_metrics': metrics_to_dict(report.positive_only_metrics),
        'stratified_reports': [
            {
                'stratum': sr.stratum.value,
                'document_count': sr.document_count,
                'metrics': metrics_to_dict(sr.metrics)
            }
            for sr in report.stratified_reports
        ],
        'confidence_intervals': {
            name: ci_to_dict(ci) for name, ci in report.confidence_intervals.items()
        }
    }

    with open(output_filepath, 'w', encoding='utf-8') as f:
        json.dump(output_data, f, indent=2)

---
EVALUATE FOR DEVELOPMENT
---

In [10]:
##SET PATHS

BASE_DIR = Path("/content/drive/MyDrive/AI6129")
GROUND_TRUTH_FILE = BASE_DIR / "accession/validation_splits/validation_splits.json"
PREDICTIONS_FILE_DEV = BASE_DIR / "accession/experiment_outputs/claude-haiku-4.5_development_predictions.json"
OUTPUT_DIR = BASE_DIR / "accession/evaluation_outputs"

# Create output directory
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Ground truth: {GROUND_TRUTH_FILE}")
print(f"Predictions: {PREDICTIONS_FILE_DEV}")
print(f"Output dir: {OUTPUT_DIR}")

Ground truth: /content/drive/MyDrive/AI6129/accession/validation_splits/validation_splits.json
Predictions: /content/drive/MyDrive/AI6129/accession/experiment_outputs/claude-haiku-4.5_development_predictions.json
Output dir: /content/drive/MyDrive/AI6129/accession/evaluation_outputs


In [11]:
datestamp = datetime.now().strftime("%Y%m%d")

# Load ground truth for development split
ground_truth = load_ground_truth(str(GROUND_TRUTH_FILE), "development")
print(f"Loaded {len(ground_truth)} ground truth documents")

# Load predictions
predictions, metadata = load_predictions(str(PREDICTIONS_FILE_DEV))
print(f"Loaded {len(predictions)} predictions")
print(f"Model: {metadata.get('model_id')}")
print(f"Total cost: ${metadata.get('total_cost_usd', 0):.4f}")

# Generate evaluation report
report_dev = evaluate_single_model(
    predictions_filepath=str(PREDICTIONS_FILE_DEV),
    ground_truth_filepath=str(GROUND_TRUTH_FILE),
    split_name="development"
)

print(f"\n=== Evaluation Results: {report_dev.model_id} ===")
print(f"Split: {report_dev.split_name}")
print(f"Documents: {report_dev.overall_metrics.document_count}")

# Overall metrics
print("\n=== Overall Metrics ===")
print(f"Macro Precision: {report_dev.overall_metrics.macro_precision:.4f}")
print(f"Macro Recall:    {report_dev.overall_metrics.macro_recall:.4f}")
print(f"Macro F1:        {report_dev.overall_metrics.macro_f1:.4f}")
print(f"Micro F1:        {report_dev.overall_metrics.micro_f1:.4f}")
print(f"Exact Match:     {report_dev.overall_metrics.exact_match_rate:.4f}")
print(f"Hallucination:   {report_dev.overall_metrics.hallucination_rate:.4f}")

# Confidence intervals
print("\n=== 95% Confidence Intervals ===")
for metric_name, ci in report_dev.confidence_intervals.items():
    print(f"{metric_name}: {ci.point_estimate:.4f} [{ci.lower_bound:.4f}, {ci.upper_bound:.4f}]")

# Stratified performance
print("\n=== Stratified Performance ===")
for strat_report in report_dev.stratified_reports:
    print(f"\n{strat_report.stratum.value.upper()} stratum ({strat_report.document_count} docs):")
    print(f"  Macro F1:       {strat_report.metrics.macro_f1:.4f}")
    print(f"  Exact Match:    {strat_report.metrics.exact_match_rate:.4f}")
    print(f"  Hallucination:  {strat_report.metrics.hallucination_rate:.4f}")

# Generate summary DataFrame
summary_df = generate_summary_table([report_dev])
print("\n=== Summary Table ===")
display(summary_df)

# Stratified table
stratified_df = generate_stratified_table([report_dev])
print("\n=== Stratified Table ===")
display(stratified_df)

# Save evaluation report
report_filepath_dev = OUTPUT_DIR / f"{report_dev.model_id}_{report_dev.split_name}_evaluation_{datestamp}.json"
save_evaluation_report(report_dev, str(report_filepath_dev))
print(f"Evaluation saved to: {report_filepath_dev}")

# Save summary tables as CSV
summary_df.to_csv(OUTPUT_DIR / f"summary_metrics_devt_{datestamp}.csv", index=False)
stratified_df.to_csv(OUTPUT_DIR / f"stratified_metrics_devt_{datestamp}.csv", index=False)
print("CSV tables saved")

Loaded 45 ground truth documents
Loaded 45 predictions
Model: claude-haiku-4.5
Total cost: $1.0625

=== Evaluation Results: claude-haiku-4.5 ===
Split: development
Documents: 45

=== Overall Metrics ===
Macro Precision: 0.9503
Macro Recall:    0.6376
Macro F1:        0.6816
Micro F1:        0.8749
Exact Match:     0.3333
Hallucination:   0.2000

=== 95% Confidence Intervals ===
macro_f1: 0.6816 [0.5651, 0.7806]
exact_match_rate: 0.3333 [0.2000, 0.4667]
hallucination_rate: 0.2000 [0.0889, 0.3111]

=== Stratified Performance ===

ZERO stratum (11 docs):
  Macro F1:       1.0000
  Exact Match:    1.0000
  Hallucination:  0.0000

LOW stratum (8 docs):
  Macro F1:       0.2500
  Exact Match:    0.2500
  Hallucination:  0.0000

MEDIUM stratum (9 docs):
  Macro F1:       0.7337
  Exact Match:    0.1111
  Hallucination:  0.1111

HIGH stratum (17 docs):
  Macro F1:       0.6512
  Exact Match:    0.0588
  Hallucination:  0.4706

=== Summary Table ===


,model_id,split,n_documents,macro_precision,macro_recall,macro_f1,micro_f1,exact_match_rate,hallucination_rate,f1_ci_lower,f1_ci_upper
0,claude-haiku-4.5,development,45,0.950289,0.637564,0.681643,0.874934,0.333333,0.2,0.565066,0.780588



=== Stratified Table ===


,model_id,stratum,n_documents,macro_f1,exact_match_rate,hallucination_rate
0,claude-haiku-4.5,zero,11,1.000000,1.000000,0.000000
1,claude-haiku-4.5,low,8,0.250000,0.250000,0.000000
2,claude-haiku-4.5,medium,9,0.733686,0.111111,0.111111
3,claude-haiku-4.5,high,17,0.651220,0.058824,0.470588


Evaluation saved to: /content/drive/MyDrive/AI6129/accession/evaluation_outputs/claude-haiku-4.5_development_evaluation_20260124.json
CSV tables saved


---
Evaluate for Validation
---


In [12]:
# Set paths
BASE_DIR = Path("/content/drive/MyDrive/AI6129")
GROUND_TRUTH_FILE = BASE_DIR / "accession/validation_splits/validation_splits.json"
PREDICTIONS_FILE_VAL = BASE_DIR / "accession/experiment_outputs/claude-haiku-4.5_validation_predictions.json"
OUTPUT_DIR = BASE_DIR / "accession/evaluation_outputs"

# Create output directory
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Ground truth: {GROUND_TRUTH_FILE}")
print(f"Predictions: {PREDICTIONS_FILE_VAL}")
print(f"Output dir: {OUTPUT_DIR}")

Ground truth: /content/drive/MyDrive/AI6129/accession/validation_splits/validation_splits.json
Predictions: /content/drive/MyDrive/AI6129/accession/experiment_outputs/claude-haiku-4.5_validation_predictions.json
Output dir: /content/drive/MyDrive/AI6129/accession/evaluation_outputs


In [13]:
datestamp = datetime.now().strftime("%Y%m%d")

# Load ground truth for validation split
ground_truth = load_ground_truth(str(GROUND_TRUTH_FILE), "validation")
print(f"Loaded {len(ground_truth)} ground truth documents")

# Load predictions
predictions, metadata = load_predictions(str(PREDICTIONS_FILE_VAL))
print(f"Loaded {len(predictions)} predictions")
print(f"Model: {metadata.get('model_id')}")
print(f"Total cost: ${metadata.get('total_cost_usd', 0):.4f}")

# Generate evaluation report
report_val = evaluate_single_model(
    predictions_filepath=str(PREDICTIONS_FILE_VAL),
    ground_truth_filepath=str(GROUND_TRUTH_FILE),
    split_name="validation"
)

print(f"\n=== Evaluation Results: {report_val.model_id} ===")
print(f"Split: {report_val.split_name}")
print(f"Documents: {report_val.overall_metrics.document_count}")

# Overall metrics
print("\n=== Overall Metrics ===")
print(f"Macro Precision: {report_val.overall_metrics.macro_precision:.4f}")
print(f"Macro Recall:    {report_val.overall_metrics.macro_recall:.4f}")
print(f"Macro F1:        {report_val.overall_metrics.macro_f1:.4f}")
print(f"Micro F1:        {report_val.overall_metrics.micro_f1:.4f}")
print(f"Exact Match:     {report_val.overall_metrics.exact_match_rate:.4f}")
print(f"Hallucination:   {report_val.overall_metrics.hallucination_rate:.4f}")

# Confidence intervals
print("\n=== 95% Confidence Intervals ===")
for metric_name, ci in report_val.confidence_intervals.items():
    print(f"{metric_name}: {ci.point_estimate:.4f} [{ci.lower_bound:.4f}, {ci.upper_bound:.4f}]")

# Stratified performance
print("\n=== Stratified Performance ===")
for strat_report in report_val.stratified_reports:
    print(f"\n{strat_report.stratum.value.upper()} stratum ({strat_report.document_count} docs):")
    print(f"  Macro F1:       {strat_report.metrics.macro_f1:.4f}")
    print(f"  Exact Match:    {strat_report.metrics.exact_match_rate:.4f}")
    print(f"  Hallucination:  {strat_report.metrics.hallucination_rate:.4f}")

# Generate summary DataFrame
summary_df = generate_summary_table([report_val])
print("\n=== Summary Table ===")
display(summary_df)

# Stratified table
stratified_df = generate_stratified_table([report_val])
print("\n=== Stratified Table ===")
display(stratified_df)

# Save evaluation report
report_filepath_val = OUTPUT_DIR / f"{report_val.model_id}_{report_val.split_name}_evaluation_{datestamp}.json"
save_evaluation_report(report_val, str(report_filepath_val))
print(f"Evaluation saved to: {report_filepath_val}")

# Save summary tables as CSV
summary_df.to_csv(OUTPUT_DIR / f"summary_metrics_val_{datestamp}.csv", index=False)
stratified_df.to_csv(OUTPUT_DIR / f"stratified_metrics_val_{datestamp}.csv", index=False)
print("CSV tables saved")

Loaded 117 ground truth documents
Loaded 117 predictions
Model: claude-haiku-4.5
Total cost: $2.4836

=== Evaluation Results: claude-haiku-4.5 ===
Split: validation
Documents: 117

=== Overall Metrics ===
Macro Precision: 0.9180
Macro Recall:    0.7193
Macro F1:        0.7448
Micro F1:        0.7333
Exact Match:     0.4274
Hallucination:   0.2308

=== 95% Confidence Intervals ===
macro_f1: 0.7448 [0.6801, 0.8130]
exact_match_rate: 0.4274 [0.3504, 0.5299]
hallucination_rate: 0.2308 [0.1538, 0.3077]

=== Stratified Performance ===

ZERO stratum (30 docs):
  Macro F1:       1.0000
  Exact Match:    1.0000
  Hallucination:  0.0000

LOW stratum (21 docs):
  Macro F1:       0.4762
  Exact Match:    0.3810
  Hallucination:  0.0952

MEDIUM stratum (24 docs):
  Macro F1:       0.6767
  Exact Match:    0.2500
  Hallucination:  0.2083

HIGH stratum (42 docs):
  Macro F1:       0.7359
  Exact Match:    0.1429
  Hallucination:  0.4762

=== Summary Table ===


,model_id,split,n_documents,macro_precision,macro_recall,macro_f1,micro_f1,exact_match_rate,hallucination_rate,f1_ci_lower,f1_ci_upper
0,claude-haiku-4.5,validation,117,0.917958,0.719272,0.744842,0.733333,0.42735,0.230769,0.680063,0.812969



=== Stratified Table ===


,model_id,stratum,n_documents,macro_f1,exact_match_rate,hallucination_rate
0,claude-haiku-4.5,zero,30,1.000000,1.000000,0.000000
1,claude-haiku-4.5,low,21,0.476190,0.380952,0.095238
2,claude-haiku-4.5,medium,24,0.676687,0.250000,0.208333
3,claude-haiku-4.5,high,42,0.735857,0.142857,0.476190


Evaluation saved to: /content/drive/MyDrive/AI6129/accession/evaluation_outputs/claude-haiku-4.5_validation_evaluation_20260124.json
CSV tables saved


---
Evaluate for Test
---


In [15]:
# Set paths
BASE_DIR = Path("/content/drive/MyDrive/AI6129")
GROUND_TRUTH_FILE = BASE_DIR / "accession/validation_splits/validation_splits.json"
PREDICTIONS_FILE_TEST = BASE_DIR / "accession/experiment_outputs/claude-haiku-4.5_test_predictions.json"
OUTPUT_DIR = BASE_DIR / "accession/evaluation_outputs"

# Create output directory
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Ground truth: {GROUND_TRUTH_FILE}")
print(f"Predictions: {PREDICTIONS_FILE_TEST}")
print(f"Output dir: {OUTPUT_DIR}")

Ground truth: /content/drive/MyDrive/AI6129/accession/validation_splits/validation_splits.json
Predictions: /content/drive/MyDrive/AI6129/accession/experiment_outputs/claude-haiku-4.5_test_predictions.json
Output dir: /content/drive/MyDrive/AI6129/accession/evaluation_outputs


In [16]:
datestamp = datetime.now().strftime("%Y%m%d")

# Load ground truth for validation split
ground_truth = load_ground_truth(str(GROUND_TRUTH_FILE), "test")
print(f"Loaded {len(ground_truth)} ground truth documents")

# Load predictions
predictions, metadata = load_predictions(str(PREDICTIONS_FILE_TEST))
print(f"Loaded {len(predictions)} predictions")
print(f"Model: {metadata.get('model_id')}")
print(f"Total cost: ${metadata.get('total_cost_usd', 0):.4f}")

# Generate evaluation report
report_test = evaluate_single_model(
    predictions_filepath=str(PREDICTIONS_FILE_TEST),
    ground_truth_filepath=str(GROUND_TRUTH_FILE),
    split_name="test"
)

print(f"\n=== Evaluation Results: {report_test.model_id} ===")
print(f"Split: {report_test.split_name}")
print(f"Documents: {report_test.overall_metrics.document_count}")

# Overall metrics
print("\n=== Overall Metrics ===")
print(f"Macro Precision: {report_test.overall_metrics.macro_precision:.4f}")
print(f"Macro Recall:    {report_test.overall_metrics.macro_recall:.4f}")
print(f"Macro F1:        {report_test.overall_metrics.macro_f1:.4f}")
print(f"Micro F1:        {report_test.overall_metrics.micro_f1:.4f}")
print(f"Exact Match:     {report_test.overall_metrics.exact_match_rate:.4f}")
print(f"Hallucination:   {report_test.overall_metrics.hallucination_rate:.4f}")

# Confidence intervals
print("\n=== 95% Confidence Intervals ===")
for metric_name, ci in report_test.confidence_intervals.items():
    print(f"{metric_name}: {ci.point_estimate:.4f} [{ci.lower_bound:.4f}, {ci.upper_bound:.4f}]")

# Stratified performance
print("\n=== Stratified Performance ===")
for strat_report in report_test.stratified_reports:
    print(f"\n{strat_report.stratum.value.upper()} stratum ({strat_report.document_count} docs):")
    print(f"  Macro F1:       {strat_report.metrics.macro_f1:.4f}")
    print(f"  Exact Match:    {strat_report.metrics.exact_match_rate:.4f}")
    print(f"  Hallucination:  {strat_report.metrics.hallucination_rate:.4f}")

# Generate summary DataFrame
summary_df = generate_summary_table([report_test])
print("\n=== Summary Table ===")
display(summary_df)

# Stratified table
stratified_df = generate_stratified_table([report_test])
print("\n=== Stratified Table ===")
display(stratified_df)

# Save evaluation report
report_filepath_test = OUTPUT_DIR / f"{report_test.model_id}_{report_test.split_name}_evaluation_{datestamp}.json"
save_evaluation_report(report_test, str(report_filepath_test))
print(f"Evaluation saved to: {report_filepath_test}")

# Save summary tables as CSV

summary_df.to_csv(OUTPUT_DIR / f"summary_metrics_test_{datestamp}.csv", index=False)
stratified_df.to_csv(OUTPUT_DIR / f"stratified_metrics_test_{datestamp}.csv", index=False)
print("CSV tables saved")

Loaded 65 ground truth documents
Loaded 65 predictions
Model: claude-haiku-4.5
Total cost: $1.2055

=== Evaluation Results: claude-haiku-4.5 ===
Split: test
Documents: 65

=== Overall Metrics ===
Macro Precision: 0.9358
Macro Recall:    0.7263
Macro F1:        0.7456
Micro F1:        0.8865
Exact Match:     0.4462
Hallucination:   0.1846

=== 95% Confidence Intervals ===
macro_f1: 0.7456 [0.6550, 0.8323]
exact_match_rate: 0.4462 [0.3231, 0.5846]
hallucination_rate: 0.1846 [0.0923, 0.2769]

=== Stratified Performance ===

ZERO stratum (18 docs):
  Macro F1:       1.0000
  Exact Match:    1.0000
  Hallucination:  0.0000

LOW stratum (11 docs):
  Macro F1:       0.4545
  Exact Match:    0.4545
  Hallucination:  0.0000

MEDIUM stratum (13 docs):
  Macro F1:       0.6121
  Exact Match:    0.1538
  Hallucination:  0.2308

HIGH stratum (23 docs):
  Macro F1:       0.7611
  Exact Match:    0.1739
  Hallucination:  0.3913

=== Summary Table ===


,model_id,split,n_documents,macro_precision,macro_recall,macro_f1,micro_f1,exact_match_rate,hallucination_rate,f1_ci_lower,f1_ci_upper
0,claude-haiku-4.5,test,65,0.935824,0.726325,0.745584,0.886473,0.446154,0.184615,0.655036,0.83234



=== Stratified Table ===


,model_id,stratum,n_documents,macro_f1,exact_match_rate,hallucination_rate
0,claude-haiku-4.5,zero,18,1.000000,1.000000,0.000000
1,claude-haiku-4.5,low,11,0.454545,0.454545,0.000000
2,claude-haiku-4.5,medium,13,0.612088,0.153846,0.230769
3,claude-haiku-4.5,high,23,0.761122,0.173913,0.391304


Evaluation saved to: /content/drive/MyDrive/AI6129/accession/evaluation_outputs/claude-haiku-4.5_test_evaluation_20260124.json
CSV tables saved


---
Analysing Documents with Errors
---

In [17]:
# Find documents with errors for analysis for Dev
print("\n=== Error Analysis ===")

# False positives (hallucinations)
hallucination_docs = [dm for dm in report_dev.document_metrics if dm.has_hallucination]
print(f"\nDocuments with hallucinations: {len(hallucination_docs)}")
for dm in hallucination_docs:  # Show first 5
    print(f"  {dm.pmcid}: {dm.false_positives} false positives")

# Missed accessions (low recall)
low_recall_docs = [dm for dm in report_dev.document_metrics
                   if dm.recall < 0.5 and dm.false_negatives > 0]
print(f"\nDocuments with low recall (<50%): {len(low_recall_docs)}")
for dm in sorted(low_recall_docs, key=lambda x: x.false_negatives, reverse=True)[:5]:
    print(f"  {dm.pmcid}: missed {dm.false_negatives} accessions (recall: {dm.recall:.2f})")


=== Error Analysis ===

Documents with hallucinations: 9
  PMC9219948: 1 false positives
  PMC5544996: 1 false positives
  PMC9493365: 1 false positives
  PMC9708683: 1 false positives
  PMC7064254: 1 false positives
  PMC8557873: 1 false positives
  PMC7433231: 54 false positives
  PMC9119036: 2 false positives
  PMC9409446: 5 false positives

Documents with low recall (<50%): 14
  PMC7433231: missed 54 accessions (recall: 0.04)
  PMC9708683: missed 33 accessions (recall: 0.28)
  PMC9754226: missed 18 accessions (recall: 0.00)
  PMC7643746: missed 6 accessions (recall: 0.14)
  PMC4748464: missed 5 accessions (recall: 0.17)


In [18]:
# Find documents with errors for analysis for Validation
print("\n=== Error Analysis ===")

# False positives (hallucinations)
hallucination_docs = [dm for dm in report_val.document_metrics if dm.has_hallucination]
print(f"\nDocuments with hallucinations: {len(hallucination_docs)}")
for dm in hallucination_docs:  # Show first 5
    print(f"  {dm.pmcid}: {dm.false_positives} false positives")

# Missed accessions (low recall)
low_recall_docs = [dm for dm in report_val.document_metrics
                   if dm.recall < 0.5 and dm.false_negatives > 0]
print(f"\nDocuments with low recall (<50%): {len(low_recall_docs)}")
for dm in sorted(low_recall_docs, key=lambda x: x.false_negatives, reverse=True)[:5]:
    print(f"  {dm.pmcid}: missed {dm.false_negatives} accessions (recall: {dm.recall:.2f})")


=== Error Analysis ===

Documents with hallucinations: 27
  PMC9309525: 2 false positives
  PMC7694379: 1 false positives
  PMC7240076: 1 false positives
  PMC6378779: 1 false positives
  PMC7425045: 2 false positives
  PMC7745040: 1 false positives
  PMC5799193: 1 false positives
  PMC8175864: 10 false positives
  PMC8471515: 3 false positives
  PMC7011100: 2 false positives
  PMC7478631: 2 false positives
  PMC9393422: 1 false positives
  PMC7557518: 1 false positives
  PMC8145149: 3 false positives
  PMC4932652: 9 false positives
  PMC9323645: 4 false positives
  PMC6703130: 3 false positives
  PMC8601536: 1 false positives
  PMC7530935: 2 false positives
  PMC9670943: 2 false positives
  PMC6123440: 1 false positives
  PMC9600463: 2 false positives
  PMC8953408: 1 false positives
  PMC7385254: 4 false positives
  PMC7291093: 18 false positives
  PMC6667439: 4 false positives
  PMC8018540: 5 false positives

Documents with low recall (<50%): 23
  PMC7441611: missed 63 accessions (r

In [19]:
# Find documents with errors for analysis for Test
print("\n=== Error Analysis ===")

# False positives (hallucinations)
hallucination_docs = [dm for dm in report_test.document_metrics if dm.has_hallucination]
print(f"\nDocuments with hallucinations: {len(hallucination_docs)}")
for dm in hallucination_docs:  # Show first 5
    print(f"  {dm.pmcid}: {dm.false_positives} false positives")

# Missed accessions (low recall)
low_recall_docs = [dm for dm in report_test.document_metrics
                   if dm.recall < 0.5 and dm.false_negatives > 0]
print(f"\nDocuments with low recall (<50%): {len(low_recall_docs)}")
for dm in sorted(low_recall_docs, key=lambda x: x.false_negatives, reverse=True)[:5]:
    print(f"  {dm.pmcid}: missed {dm.false_negatives} accessions (recall: {dm.recall:.2f})")


=== Error Analysis ===

Documents with hallucinations: 12
  PMC8253257: 1 false positives
  PMC9578181: 1 false positives
  PMC9146225: 1 false positives
  PMC6026178: 3 false positives
  PMC8549363: 4 false positives
  PMC5966662: 7 false positives
  PMC7725340: 17 false positives
  PMC8767345: 19 false positives
  PMC9251186: 1 false positives
  PMC9739812: 3 false positives
  PMC5145817: 7 false positives
  PMC7783090: 1 false positives

Documents with low recall (<50%): 16
  PMC7725340: missed 24 accessions (recall: 0.41)
  PMC8767345: missed 16 accessions (recall: 0.16)
  PMC5966662: missed 7 accessions (recall: 0.46)
  PMC4810482: missed 6 accessions (recall: 0.14)
  PMC9035464: missed 6 accessions (recall: 0.00)
